# MoNuSAC · Notebook 1 — CONCH embeddings (mirror of `conch-set-up.ipynb`)

Loads frozen CONCH, builds 64px nucleus crops from the **RationAI/MoNuSAC** HF mirror,
embeds them, and caches everything to `/kaggle/working/` for Notebook 2.

**Run on Kaggle/Colab with a GPU and an HF token that has CONCH access.**

### VERIFY 3 THINGS before the first full run (I could not introspect the live dataset):
1. **Field names** — run the `.features` cell below and set `IMAGE_FIELD / INSTANCES_FIELD / LABELS_FIELD`.
2. **Organ source** — MoNuSAC's organ may not be an explicit column. If not, fill `PATIENT_TO_ORGAN`.
3. **Mask format** — `instance_centroid()` assumes a 2D binary mask. Adapt if RLE / bbox-relative.


In [1]:
!GIT_TERMINAL_PROMPT=0 pip install -q "git+https://github.com/mahmoodlab/CONCH.git" "datasets>=2.18"

import torch
from conch.open_clip_custom import create_model_from_pretrained
# HF token: Kaggle secret or env var
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    import os; token = os.environ.get("HF_TOKEN")

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = create_model_from_pretrained("conch_ViT-B-16", "hf_hub:MahmoodLab/conch", hf_auth_token=token)
model = model.to(device).eval()
print("CONCH loaded on", device)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is inco

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


meta.yaml:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/802M [00:00<?, ?B/s]

CONCH loaded on cuda


In [2]:
# ---- inspect the mirror schema ONCE, then set the field names below ----
from datasets import load_dataset
ds = load_dataset("RationAI/MoNuSAC")
print(ds)
print(ds["train"].features)          # <-- read this, then fix the CONFIG cell

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/213M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/91.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/209 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/101 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['patient', 'image', 'instances', 'categories', 'tissue'],
        num_rows: 209
    })
    test: Dataset({
        features: ['patient', 'image', 'instances', 'categories', 'tissue'],
        num_rows: 101
    })
})
{'patient': Value('string'), 'image': Image(mode='RGB', decode=True), 'instances': List(Image(mode='1', decode=True)), 'categories': List(ClassLabel(names=['Ambiguous', 'Epithelial', 'Lymphocyte', 'Macrophage', 'Neutrophil'])), 'tissue': ClassLabel(names=['Breast', 'Kidney', 'Lung', 'Prostate'])}


In [3]:
import numpy as np

# ===================== CONFIG (verify against .features above) =====================
IMAGE_FIELD, INSTANCES_FIELD, LABELS_FIELD, PATIENT_FIELD = "image", "instances", "categories", "patient"
CROP, MIN_VALID, BATCH = 64, 0.6, 256

CLASS_NAMES     = ["Epithelial", "Lymphocyte", "Macrophage", "Neutrophil"]   # model class order
CLASS_NAMES_ALL = ["Ambiguous"] + CLASS_NAMES   # raw category order IF labels are stored as ints
DROP_LABELS     = {"Ambiguous"}

# Organ recovery: fill ONLY if there is no organ column. MoNuSAC has exactly 4 organs.
PATIENT_TO_ORGAN = {
    # "TCGA-XX-XXXX": "Breast", ...   <- from the MoNuSAC patient lists, if needed
}
ORGANS = ["Breast", "Kidney", "Lung", "Prostate"]

# ---- helpers (instance-based; replaces PanNuke's channel-based nuclei_in_image) ----
def instance_centroid(inst):
    ys, xs = np.where(np.asarray(inst) > 0)          # ADAPT if RLE/bbox
    return (int(ys.mean()), int(xs.mean())) if len(ys) else None

def crop_patch(img, r, c, size=CROP):
    h = size // 2; R, C = img.shape[:2]; r0, r1, c0, c1 = r-h, r+h, c-h, c+h
    pt, pl = max(0,-r0), max(0,-c0); pb, pr = max(0,r1-R), max(0,c1-C)
    r0, r1, c0, c1 = max(0,r0), min(R,r1), max(0,c0), min(C,c1)
    patch = np.pad(img[r0:r1, c0:c1], ((pt,pb),(pl,pr),(0,0)), "constant")
    return patch[:size, :size]

TISSUE_NAMES = ds["train"].features["tissue"].names

def resolve_organ(ex):
    return TISSUE_NAMES[ex["tissue"]]

In [4]:
for i in [0, 50, 100, 150, 200]:
    ex = ds["train"][i]
    print(ex["tissue"], resolve_organ(ex))

2 Lung
3 Prostate
2 Lung
0 Breast
3 Prostate


In [5]:
from tqdm import tqdm
name_to_idx = {n: i for i, n in enumerate(CLASS_NAMES)}

crops, labels, organs, img_idx = [], [], [], []
gid = 0
for split in ("train", "test"):                      # re-split ourselves later for grouping control
    for ex in tqdm(ds[split], desc=f"scan-{split}"):
        img = np.asarray(ex[IMAGE_FIELD].convert("RGB")).astype(np.uint8)
        organ = resolve_organ(ex)
        for inst, lab in zip(ex[INSTANCES_FIELD], ex[LABELS_FIELD]):
            lab = lab if isinstance(lab, str) else CLASS_NAMES_ALL[lab]
            if lab in DROP_LABELS or lab not in name_to_idx: continue
            ctr = instance_centroid(inst)
            if ctr is None: continue
            r, c = ctr; h = CROP // 2
            vh = min(r+h, img.shape[0]) - max(r-h, 0); vw = min(c+h, img.shape[1]) - max(c-h, 0)
            if (vh*vw)/(CROP*CROP) < MIN_VALID: continue
            crops.append(crop_patch(img, r, c)); labels.append(name_to_idx[lab])
            organs.append(organ); img_idx.append(gid)
        gid += 1

crops  = np.stack(crops)
labels = np.array(labels); organs = np.array(organs); img_idx = np.array(img_idx)
print(f"{len(crops)} nuclei | class counts: {dict(zip(CLASS_NAMES, np.bincount(labels, minlength=4)))}")
print("organ counts:", dict(zip(*np.unique(organs, return_counts=True))))
assert "UNKNOWN" not in organs, "Organ recovery failed -> fill PATIENT_TO_ORGAN (VERIFY item 2)."

scan-test: 100%|██████████| 101/101 [01:33<00:00,  1.08it/s]


46686 nuclei | class counts: {'Epithelial': np.int64(21721), 'Lymphocyte': np.int64(23281), 'Macrophage': np.int64(893), 'Neutrophil': np.int64(791)}
organ counts: {np.str_('Breast'): np.int64(12732), np.str_('Kidney'): np.int64(12367), np.str_('Lung'): np.int64(9561), np.str_('Prostate'): np.int64(12026)}


In [6]:
from PIL import Image
# ---- frozen CONCH embeddings: probe (pre-projection) + contrast (kept for completeness) ----
emb_probe, emb_contrast = [], []
for s in tqdm(range(0, len(crops), BATCH), desc="embed"):
    t = torch.stack([preprocess(Image.fromarray(p)) for p in crops[s:s+BATCH]]).to(device)
    with torch.inference_mode():
        emb_probe.append(model.encode_image(t, proj_contrast=False, normalize=False).cpu().numpy())
        emb_contrast.append(model.encode_image(t, proj_contrast=True, normalize=True).cpu().numpy())
emb_probe = np.concatenate(emb_probe); emb_contrast = np.concatenate(emb_contrast)

np.savez_compressed("/kaggle/working/monusac_conch.npz",
    emb_probe=emb_probe, emb_contrast=emb_contrast,
    labels=labels, organs=organs, img_idx=img_idx)
# crops cached separately so Notebook 2's appearance arm + gate can re-jitter without re-scanning:
np.save("/kaggle/working/monusac_crops.npy", crops)
print("probe:", emb_probe.shape, "| cached. NOW SAVE A VERSION so the output persists.")

embed: 100%|██████████| 183/183 [1:25:30<00:00, 28.04s/it]


probe: (46686, 512) | cached. NOW SAVE A VERSION so the output persists.


In [7]:
# ---- quick probe sanity check (image-grouped split) ----
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import balanced_accuracy_score, classification_report

rng = np.random.RandomState(0); imgs = np.unique(img_idx); rng.shuffle(imgs)
train_imgs = set(imgs[:int(0.75*len(imgs))].tolist())
tr = np.array([i in train_imgs for i in img_idx])
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
clf.fit(emb_probe[tr], labels[tr]); pred = clf.predict(emb_probe[~tr])
print("Image-grouped balanced accuracy:", round(balanced_accuracy_score(labels[~tr], pred), 3))
print(classification_report(labels[~tr], pred, target_names=CLASS_NAMES, digits=3))

Image-grouped balanced accuracy: 0.81
              precision    recall  f1-score   support

  Epithelial      0.958     0.882     0.919      5076
  Lymphocyte      0.927     0.931     0.929      5894
  Macrophage      0.544     0.685     0.606       289
  Neutrophil      0.282     0.741     0.409       189

    accuracy                          0.900     11448
   macro avg      0.678     0.810     0.716     11448
weighted avg      0.921     0.900     0.908     11448

